# Hands-on: temperature anomalies with CDO and NCO
**Yiwen Li · Fresh Eyes on CMIP**

Use real ACCESS-CM2 historical + SSP5-8.5 data to reproduce the 1979–2024 result,
then change one scientific choice at a time. This notebook runs independently.

Run every cell in order with **Shift+Enter**, including collapsed cells. All CDO
and NCO processing commands are shown below their explanations.

## 02 · The calculation we are comparing

| Step | Definition used by every route |
|---|---|
| Input | ACCESS-CM2 monthly `tas` in Kelvin, historical + SSP5-8.5; or explicitly labeled synthetic test data |
| Main period | January 1979–December 2024: **552 months, 46 annual values** |
| Reference | January 1981–December 2010: 30 values for each calendar month |
| Monthly anomaly | Temperature minus the matching calendar-month reference field |
| Mean anomaly map | **Month-duration weighted** mean over the analysis period |
| Arctic mean | Latitude centres 65–90°N, all longitudes, **equal grid-point weights** |
| Annual stripes | **Equal weight for the 12 monthly samples** in each year |
| Comparison | Maximum absolute difference ≤ 0.0003°C, relative tolerance 0; coordinates and calendar labels agree |

The map weights each monthly field by its duration. The annual series gives
each month equal weight. Keeping these definitions the same across tools makes
the numerical comparison meaningful.

In [ ]:
# Select the data and create this run
from pathlib import Path
import os, sys, subprocess, json, uuid
from datetime import datetime, timezone
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts/run_python_updated.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/run_python_updated.py").exists(), "Open this notebook from the workshop repository."
os.chdir(ROOT)

# The seminar uses real ACCESS-CM2. Synthetic mode is an explicit software-test option.
DATA_MODE = os.environ.get("WORKSHOP_MODE", "real")  # "real" or "synthetic"
END_YEAR = int(os.environ.get("WORKSHOP_END_YEAR", "2024"))
assert DATA_MODE in {"synthetic", "real"}
assert END_YEAR >= 2024
os.environ["OMP_NUM_THREADS"] = "1"
if DATA_MODE == "synthetic":
    fixture = ROOT / "data/updated_fixture"
    subprocess.run([sys.executable, "scripts/generate_updated_fixture.py", str(fixture)], check=True)
    HIST, FUTURE = fixture / "historical.nc", fixture / "ssp585.nc"
else:
    directory = ROOT / "data/access_cm2"
    HIST = Path(os.environ.get("WORKSHOP_HISTORICAL", str(directory / "tas_Amon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc")))
    FUTURE = Path(os.environ.get("WORKSHOP_FUTURE", str(directory / "tas_Amon_ACCESS-CM2_ssp585_r1i1p1f1_gn_201501-210012.nc")))
    missing = [str(p) for p in (HIST, FUTURE) if not p.is_file()]
    if missing:
        raise FileNotFoundError("Real-data mode requires both raw files. No synthetic substitution is allowed. Missing: " + "; ".join(missing) + ". See docs/UPDATED_DEMO.md.")

RUN = ROOT / "outputs" / ("updated_" + DATA_MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True)
for route in ("cdo", "nco"):
    directory = RUN / route
    (directory / "work").mkdir(parents=True)
    (directory / "historical_input.nc").symlink_to(HIST.resolve())
    (directory / "ssp585_input.nc").symlink_to(FUTURE.resolve())
os.environ.update(DEMO_HIST=os.path.relpath(HIST, ROOT), DEMO_FUTURE=os.path.relpath(FUTURE, ROOT), DEMO_OUT=os.path.relpath(RUN, ROOT), DEMO_END=str(END_YEAR))
print("DATA MODE:", "SYNTHETIC SOFTWARE TEST — NOT ACCESS-CM2" if DATA_MODE == "synthetic" else "REAL ACCESS-CM2: historical + SSP5-8.5")
print("Analysis:", f"1979–{END_YEAR}", "| Monthly reference: 1981–2010")
print("Outputs:", RUN.relative_to(ROOT))

In [ ]:
# Check data coverage and tool versions
_ = subprocess.run([sys.executable, "scripts/check_updated_inputs.py", str(HIST), str(FUTURE), "--end-year", str(END_YEAR)], check=True)
for command in (["cdo", "-V"], ["ncks", "--version"]):
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    print((result.stdout + result.stderr).splitlines()[0])
print("Python", sys.version.split()[0])

## 1 · Reproduce the common result

Keep the baseline, region and time weights fixed for the first calculation.
The independent reference and both command-line routes start from the raw data.

In [ ]:
# Calculate the independent reference
_ = subprocess.run([sys.executable, "scripts/run_python_updated.py", str(HIST), str(FUTURE), str(RUN / "python"), "--end-year", str(END_YEAR)], check=True, capture_output=True, text=True)
print(f"Reference ready: five products for 1979–{END_YEAR}.")

Each cell below contains the commands it executes. Use **Shift+Enter** to run
one cell and inspect its result before continuing.

- `%%bash` runs shell commands in this notebook.
- `cd "$DEMO_OUT/cdo"` or `cd "$DEMO_OUT/nco"` enters that route's result folder.
- `historical_input.nc` and `ssp585_input.nc` are short names for the two original files.
- `$DEMO_END` is 2024; it can be changed for the optional extension.
- `-O` permits replacing an output file when a cell is rerun. `set -euo pipefail`
  stops a cell if a command fails.

## 1A · CDO

### 1A.01 · Select temperature and years

Input: the historical and SSP585 files. Output: `work/historical.nc`
and `work/ssp585.nc`, containing only monthly near-surface air temperature
(`tas`) for 1979–2014 and 2015–2024. `DEMO_END` is 2024 for this exercise.
`selname` selects the variable; `selyear` selects the dates.

Each command reads its input file and writes its last file argument.
The common options allow reruns (`-O`), use sequential I/O (`-L`), reduce log
messages (`-s`), and write double-precision classic NetCDF (`-b F64 -f nc`).

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc selname,tas historical_input.nc work/historical_tas.nc
cdo -O -L -s -b F64 -f nc selyear,1979/2014 work/historical_tas.nc work/historical.nc
cdo -O -L -s -b F64 -f nc selname,tas ssp585_input.nc work/ssp585_tas.nc
cdo -O -L -s -b F64 -f nc selyear,2015/"$DEMO_END" work/ssp585_tas.nc work/ssp585.nc

### 1A.02 · Join the two experiments

Input: the two selected temperature files. Output: `work/merged.nc`.
`mergetime` joins and sorts their monthly records into the continuous
1979–2024 series: 552 months. Both experiments use the same ACCESS-CM2
member and grid.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc mergetime work/historical.nc work/ssp585.nc work/merged.nc

### 1A.03 · Convert Kelvin to Celsius

Input: `work/merged.nc`, with temperature in Kelvin.
Output: `work/celsius.nc`, in degrees Celsius.
`subc,273.15` changes the values; `setunit,degC` updates the unit label.
Changing the label alone would leave the values in Kelvin.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc subc,273.15 work/merged.nc work/celsius_raw.nc
cdo -O -L -s -b F64 -f nc setunit,degC work/celsius_raw.nc work/celsius.nc

### 1A.04 · Select the reference period

Input: `work/celsius.nc`. Output: `work/baseline.nc`.
Select 1981–2010, the fixed 30-year reference period, giving 360 months.
This reference period stays fixed when the analysis is extended.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc selyear,1981/2010 work/celsius.nc work/baseline.nc

### 1A.05 · Calculate twelve monthly reference fields

Input: `work/baseline.nc`. Output: `climatology.nc`, containing
twelve temperature fields. `ymonmean` averages the 30 Januaries together,
then the 30 Februaries, and so on. Each matching calendar month has equal
weight; January and July have different reference fields.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc ymonmean work/baseline.nc climatology.nc

### 1A.06 · Subtract the matching monthly reference

Inputs: the full Celsius series and the twelve reference fields.
Output: `anomaly.nc`, with 552 monthly anomaly fields.
`ymonsub` subtracts the matching calendar-month reference at every grid
point: January minus reference January, February minus reference February,
and so on. `chname` renames `tas` to `tas_anom`; an anomaly is measured in
degrees Celsius, with positive values warmer than that month's reference.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc ymonsub work/celsius.nc climatology.nc work/anomaly_raw.nc
cdo -O -L -s -b F64 -f nc chname,tas,tas_anom work/anomaly_raw.nc anomaly.nc

### 1A.07 · Average the map with month-length weights

Input: `anomaly.nc`. Output: `map_mean.nc`, one mean anomaly map.
At each grid point, calculate **sum(anomaly × days in month) / sum(days in
month)**. This follows the upstream recipe's duration weighting.

`muldpm` multiplies by each month's number of days and `timsum` sums over
time. A field of ones gives the denominator using the same calendar,
including leap years. The input checks require complete monthly bounds
and valid data throughout, so this denominator covers the same samples.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc muldpm anomaly.nc work/weighted.nc
cdo -O -L -s -b F64 -f nc timsum work/weighted.nc work/weighted_sum.nc
cdo -O -L -s -b F64 -f nc expr,'tas_anom=1;' anomaly.nc work/ones.nc
cdo -O -L -s -b F64 -f nc muldpm work/ones.nc work/days.nc
cdo -O -L -s -b F64 -f nc timsum work/days.nc work/weight_sum.nc
cdo -O -L -s -b F64 -f nc div work/weighted_sum.nc work/weight_sum.nc map_mean.nc

### 1A.08 · Average the Arctic grid points

Input: `anomaly.nc`. Output: `arctic_monthly.nc`, one anomaly per
month. `sellonlatbox` selects all longitudes and grid-point latitudes from
65°N to 90°N. `fldmean,weights=false` gives each selected grid point equal
weight. On this complete rectangular grid, that matches the recipe's
unweighted longitude mean followed by its unweighted latitude mean.
This is an unweighted regional mean, not an area-weighted mean.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc sellonlatbox,0,360,65,90 anomaly.nc work/arctic.nc
cdo -O -L -s -b F64 -f nc fldmean,weights=false work/arctic.nc arctic_monthly.nc

### 1A.09 · Average the twelve months in each year

Input: `arctic_monthly.nc`. Output: `arctic_annual.nc`, one Arctic
anomaly for each of the 46 years. `yearmean` gives the twelve monthly
samples equal weight, matching the upstream annual-statistics definition.
The annual series therefore uses equal month weights, while the map above
uses month-length weights.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/cdo"

cdo -O -L -s -b F64 -f nc yearmean arctic_monthly.nc arctic_annual.nc

## 1B · NCO

### 1B.01 · Select the historical and scenario months

`ncks` extracts `tas` and its time bounds from each raw input. The
historical selection contains **432 months, 1979–2014**. For an end year of
2024, the SSP5-8.5 selection contains **120 months, 2015–2024**.

`-v` selects variables; `-d time,start,end` selects dates. `--mk_rec_dmn time`
makes time the record dimension needed by `ncrcat`. `-3` writes classic NetCDF;
`-O` replaces a previous output from this step. The outputs are
`work/historical.nc` and `work/ssp585.nc`.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncks -O -3 --mk_rec_dmn time -v tas,time_bnds \
  -d time,'1979-01-01','2014-12-31' \
  historical_input.nc work/historical.nc

ncks -O -3 --mk_rec_dmn time -v tas,time_bnds \
  -d time,'2015-01-01',"$DEMO_END-12-31" \
  ssp585_input.nc work/ssp585.nc

### 1B.02 · Join the two experiments

`ncrcat` concatenates records in the supplied order: historical
first, then SSP5-8.5. `work/merged.nc` contains **552 consecutive monthly fields
for 1979–2024**. No averaging occurs in this step.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncrcat -O work/historical.nc work/ssp585.nc work/merged.nc

### 1B.03 · Convert Kelvin to Celsius

`ncap2 -s` evaluates the visible arithmetic expression. `double(tas)`
requests double precision, and subtracting **273.15** converts Kelvin to Celsius.
The result is `work/celsius.nc` with the same monthly fields.

`ncatted -a` edits attributes. The two missing-value attributes use the same
double-precision value. The time bounds are complete, so their unused fill
attribute is removed. The original `cell_measures` link refers to an external
area file; it is removed because these spatial means use equal grid-point
weights. These attribute edits do not change the valid temperature values.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncap2 -O -s '
  tas=double(tas)-273.15;
  tas@units="degC";
' work/merged.nc work/celsius.nc

ncatted -O \
  -a _FillValue,tas,o,d,1.0e20 \
  -a missing_value,tas,o,d,1.0e20 \
  -a _FillValue,time_bnds,d,, \
  -a cell_measures,tas,d,, \
  work/celsius.nc

### 1B.04 · Select the 1981–2010 reference period

The reference is taken from the historical years in
`work/celsius.nc`. `work/baseline.nc` contains **360 consecutive months**,
starting in January 1981 and ending in December 2010.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncks -O -d time,'1981-01-01','2010-12-31' \
  work/celsius.nc work/baseline.nc

### 1B.05 · Calculate twelve monthly reference fields

NCO indices start at **zero**. For January, `m=0` and
`-d time,0,359,12` selects indices **0, 12, 24, …, 348**: the 30 Januaries.
February starts at index 1; the loop continues through December at index 11.

`ncwa -a time` takes an arithmetic mean and removes the time dimension.
`ncecat -u month` then joins the twelve fields into `climatology.nc`, preserving
January-to-December order. The final command labels the month coordinate
**1–12**. `month_files` holds only the twelve filenames.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

month_files=()
for m in {0..11}; do
  printf -v tag '%02d' "$m"
  ncwa -O -a time -d time,"$m",359,12 \
    work/baseline.nc "work/clim_$tag.nc"
  month_files+=("work/clim_$tag.nc")
done

ncecat -O -u month "${month_files[@]}" climatology.nc

ncap2 -O -s '
  month[month]=array(1,1,$month);
  month@long_name="calendar month";
' climatology.nc climatology.nc

### 1B.06 · Subtract the matching calendar-month reference

Rename the reference temperature to `tas_clim`, fix its month
dimension, and append it to the temperature file. A fixed month dimension
allows the time dimension to remain the single record dimension in classic
NetCDF.

The input starts in January and contains consecutive complete months. Therefore
`t%12` identifies the corresponding zero-based reference month for every time
index `t`. `ncap2` subtracts that reference at every latitude and longitude.
`anomaly.nc` retains **552 monthly anomaly fields** and their time bounds.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncrename -O -v tas,tas_clim climatology.nc work/clim_named.nc
ncks -O --fix_rec_dmn month work/clim_named.nc work/clim_fixed.nc
ncks -A -v tas_clim work/clim_fixed.nc work/celsius.nc

ncap2 -O -s '
  tas_anom=tas;
  for(t=0;t<$time.size;t++){
    tas_anom(t,:,:)=tas(t,:,:)-tas_clim(t%12,:,:);
  }
  tas_anom@units="degC";
  tas_anom@anomaly_reference_period="1981-2010";
' work/celsius.nc work/anomaly_all.nc

ncks -O -v tas_anom,time_bnds work/anomaly_all.nc anomaly.nc

### 1B.07 · Calculate the duration-weighted mean anomaly map

`time_bnds(:,1)-time_bnds(:,0)` gives each monthly interval's
duration in the file's time units. `ncwa -a time -w duration` computes
**Σ(anomaly × duration) / Σ(duration)** at each grid point.

`map_mean.nc` contains one latitude–longitude field. The map follows the
recipe's duration weighting and is a period-mean anomaly, not a trend.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncap2 -O -s '
  duration[time]=double(time_bnds(:,1)-time_bnds(:,0));
' anomaly.nc work/weighted.nc

ncwa -O -a time -w duration -v tas_anom \
  work/weighted.nc map_mean.nc

### 1B.08 · Average the Arctic grid points

`-d lat,65.0,90.0` selects latitude centres from 65°N to 90°N.
`ncwa -a lon` first takes the zonal mean; `ncwa -a lat` then takes the
meridional mean. No area weights are supplied, matching the recipe's arithmetic
spatial means.

`arctic_monthly.nc` contains **552 monthly Arctic anomalies**.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncwa -O -a lon -d lat,65.0,90.0 \
  anomaly.nc work/arctic_zonal.nc

ncwa -O -a lat work/arctic_zonal.nc arctic_monthly.nc

### 1B.09 · Calculate annual Arctic anomalies

`ncks --mk_rec_dmn time` prepares the monthly time series for
record averaging. With `ncra --mro -d time,,,12,12`, the stride and averaging
group both contain twelve records: January–December of each year.

`arctic_annual.nc` contains **46 annual values, 1979–2024**. Each monthly sample
has equal weight. This annual definition deliberately differs from the
duration-weighted map.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncks -O --mk_rec_dmn time \
  arctic_monthly.nc work/arctic_record.nc

ncra -O --mro -d time,,,12,12 \
  work/arctic_record.nc arctic_annual.nc

### 1B.10 · Set the annual time bounds

`ncra` also averages the monthly bounds as ordinary variables.
Replace those bounds with the start of January and the end of December for
each year, leaving the annual anomaly values unchanged.

The first selection takes records **0, 12, 24, …**; the second takes records
**11, 23, 35, …**. `-C -v` copies only the named variable, so the annual time
coordinate is preserved. The final file has **46 complete calendar-year
intervals**.

In [ ]:
%%bash
set -euo pipefail
cd "$DEMO_OUT/nco"

ncks -O -v time_bnds -d time,0,,12 \
  work/arctic_record.nc work/year_starts.nc
ncks -O -v time_bnds -d time,11,,12 \
  work/arctic_record.nc work/year_ends.nc

ncap2 -O -s 'year_end[time]=time_bnds(:,1);' \
  work/year_ends.nc work/year_ends_named.nc
ncks -A -C -v year_end work/year_ends_named.nc work/year_starts.nc

ncap2 -O -s 'time_bnds(:,1)=year_end;' \
  work/year_starts.nc work/year_bounds.nc
ncks -A -C -v time_bnds work/year_bounds.nc arctic_annual.nc

## 1C · Compare the three routes

In [ ]:
# Compare the three results
subprocess.run([sys.executable, "scripts/compare_updated_workflows.py", str(RUN)], check=True)
report = json.loads((RUN / "updated_validation.json").read_text())
assert report["passed"]
print("Maximum absolute difference (°C):", max(row["max_absolute_difference_degC"] for row in report["comparisons"].values()))
print("Ten product comparisons passed; inspect the report for coordinate, baseline and annual checks.")

In [ ]:
# Show the comparison figure
display(Image(filename=str(RUN / "updated_comparison.png")))

## 2 · Move the Arctic boundary to 70°N

Prediction: should narrowing the region change the annual anomaly? The answer
depends on the spatial structure of the data. These commands use each route's
own anomaly file. They retain equal grid-point weights and equal monthly weights.
Only the latitude threshold changes.

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$DEMO_OUT/exercises"
cdo -O -L -s -b F64 -f nc sellonlatbox,0,360,70,90   "$DEMO_OUT/cdo/anomaly.nc" "$DEMO_OUT/exercises/cdo_arctic_70N.nc"
cdo -O -L -s -b F64 -f nc fldmean,weights=false   "$DEMO_OUT/exercises/cdo_arctic_70N.nc" "$DEMO_OUT/exercises/cdo_monthly_70N.nc"
ncwa -O -a lat,lon -d lat,70.0,90.0   "$DEMO_OUT/nco/anomaly.nc" "$DEMO_OUT/exercises/nco_monthly_70N.nc"

In [ ]:
import numpy as np
import xarray as xr
with xr.open_dataset(RUN / "exercises/cdo_monthly_70N.nc") as a, xr.open_dataset(RUN / "exercises/nco_monthly_70N.nc") as b:
    cdo70 = a.tas_anom.values.squeeze()
    nco70 = b.tas_anom.values.squeeze()
    np.testing.assert_allclose(cdo70, nco70, atol=0.0003, rtol=0)
    print("70°N CDO/NCO maximum difference:", float(abs(cdo70 - nco70).max()), "°C")
with xr.open_dataset(RUN / "cdo/arctic_monthly.nc") as original:
    print("70°N versus 65°N maximum difference:", float(abs(cdo70 - original.tas_anom.values.squeeze()).max()), "°C")

## 3 · Change the annual time weights only

`yearmean` gives equal weight to each monthly sample. `yearmonmean` weights
monthly values by the number of days in each month. The latter is a valid
alternative question, but it changes the annual definition in the shared recipe.
Keep the Arctic boundary at 65°N while testing this difference.

In [ ]:
%%bash
set -euo pipefail
cdo -O -s -b F64 -f nc yearmonmean "$DEMO_OUT/cdo/arctic_monthly.nc"   "$DEMO_OUT/exercises/cdo_annual_monthlength.nc"

In [ ]:
with xr.open_dataset(RUN / "exercises/cdo_annual_monthlength.nc") as weighted, xr.open_dataset(RUN / "cdo/arctic_annual.nc") as equal:
    difference = weighted.tas_anom.values.squeeze() - equal.tas_anom.values.squeeze()
    print("Annual month-length weighting minus equal-month weighting:")
    print("Maximum absolute change:", float(abs(difference).max()), "°C")
print("The shared recipe's annual result remains the original equal-month product.")

## 4 · Check the reference period residual

For each grid point and calendar month, the average anomaly over 1981–2010
should be close to zero. This checks the month matching, not whether all
1979–2024 anomalies average to zero.

In [ ]:
%%bash
set -euo pipefail
cdo -O -s -b F64 -f nc selyear,1981/2010 "$DEMO_OUT/cdo/anomaly.nc"   "$DEMO_OUT/exercises/baseline_anomaly.nc"
cdo -O -s -b F64 -f nc ymonmean "$DEMO_OUT/exercises/baseline_anomaly.nc"   "$DEMO_OUT/exercises/baseline_residual.nc"

In [ ]:
with xr.open_dataset(RUN / "exercises/baseline_residual.nc") as residual:
    maximum = float(abs(residual.tas_anom).max())
    assert maximum <= 0.0003
    print("PASS: maximum reference-period monthly residual =", maximum, "°C")

## Optional extension · Continue the scenario to 2100

Change `WORKSHOP_END_YEAR` (or `END_YEAR` in the setup cell) to `2100`, restart,
and run all cells. Use the same historical 1981–2010 reference, and retain
explicit historical/SSP5-8.5 labeling. This produces 122 annual values, 1979–2100.
It is an extension of the seminar calculation; do not compare its period-mean
map directly with the 1979–2024 reference map as if the definitions were equal.

Save your edited notebook, NetCDF outputs, validation report and figures before
closing a Binder session. All outputs from this run are under the printed path.